# Reproduce the solar-filament predictions

Run this notebook from the released repository, with its `weights/final.pt` and `weights/provenance.json`. Attach the official competition test images or set `MAGFILO_DATA_ROOT`. This notebook performs inference only; it never reads training labels or trains a model. Missing weights, provenance, or image coverage stop execution rather than produce a substitute result.

A CUDA GPU is recommended. CPU execution is supported by the pipeline but can be substantially slower for 180 native-resolution images. Dependencies are installed only when needed; an already prepared environment can run offline. All numerical imports and model execution happen in fresh subprocesses so package installation does not reuse stale imports in the notebook kernel.

Only load the checkpoint distributed by this repository, after its SHA-256 check. The checkpoint loader uses PyTorch serialization for this trusted project asset. Reproduction establishes output agreement, not prize eligibility or a new leaderboard score.


In [ ]:
import hashlib
import importlib.metadata
import json
import os
from pathlib import Path
import subprocess
import sys

# Optional overrides: SOLARFILAMENT_ROOT, MAGFILO_DATA_ROOT, SOLARFILAMENT_OUTPUT.
# Set False in an offline environment; missing/mismatched dependencies then stop clearly.
ALLOW_INSTALL = True
EXPECTED_IMAGE_COUNT = 180
REPO_ROOT = Path(os.environ.get("SOLARFILAMENT_ROOT", Path.cwd())).resolve()
if not (REPO_ROOT / "src/solarfilament/cli.py").is_file():
    matches = [p for p in REPO_ROOT.parents if (p / "src/solarfilament/cli.py").is_file()]
    if len(matches) != 1:
        raise RuntimeError("Set SOLARFILAMENT_ROOT to the released repository directory.")
    REPO_ROOT = matches[0]
WEIGHTS = REPO_ROOT / "weights/final.pt"
PROVENANCE_PATH = REPO_ROOT / "weights/provenance.json"
for required in (WEIGHTS, PROVENANCE_PATH, REPO_ROOT / "requirements.txt"):
    if not required.is_file():
        raise FileNotFoundError(f"Required release asset is missing: {required}")

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

provenance = json.loads(PROVENANCE_PATH.read_text(encoding="utf-8"))
for key in ("weights_sha256", "submission_sha256", "config"):
    if key not in provenance:
        raise ValueError(f"Incomplete release provenance: {key}")
if sha256(WEIGHTS) != provenance["weights_sha256"]:
    raise ValueError("Weight checksum differs from the released provenance.")
default_output = Path("/kaggle/working/solarfilament-reproduction") if Path("/kaggle").exists() else REPO_ROOT / "runs/reproduction"
OUTPUT = Path(os.environ.get("SOLARFILAMENT_OUTPUT", default_output)).resolve()
OUTPUT.mkdir(parents=True, exist_ok=True)
RUN_ENV = dict(os.environ)
RUN_ENV["PYTHONPATH"] = str(REPO_ROOT / "src") + os.pathsep + RUN_ENV.get("PYTHONPATH", "")

def run_python(arguments, *, capture=False):
    result = subprocess.run(
        [sys.executable, *map(str, arguments)], cwd=REPO_ROOT, env=RUN_ENV,
        check=True, text=True, capture_output=capture,
    )
    if capture:
        return result.stdout

print(json.dumps({"repository": str(REPO_ROOT), "output": str(OUTPUT),
                  "verified_weight_sha256": provenance["weights_sha256"]}, indent=2))


In [ ]:
# Inspect distributions without importing torch/numpy into the notebook kernel.
pins = {}
for raw in (REPO_ROOT / "requirements.txt").read_text(encoding="utf-8").splitlines():
    line = raw.split("#", 1)[0].strip()
    if not line:
        continue
    if "==" not in line or line.startswith("-"):
        raise ValueError(f"Expected a pinned runtime requirement, got {line!r}")
    name, version = line.split("==", 1)
    pins[name] = version

installed = {}
for name in pins:
    try:
        installed[name] = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        installed[name] = None
stack = ("torch", "torchvision")
if any(installed.get(name) for name in stack) and not all(installed.get(name) for name in stack):
    raise RuntimeError("Install a compatible torch/torchvision pair before running this notebook.")
# Preserve the host's CUDA build. Its exact versions are recorded and verified by
# importing the pair in a fresh subprocess below; all other runtime additions are pinned.
requirements = [
    f"{name}=={version}" for name, version in pins.items()
    if name not in stack and installed.get(name) != version
]
if not all(installed.get(name) for name in stack):
    requirements += [f"{name}=={pins[name]}" for name in stack]
if requirements:
    if not ALLOW_INSTALL:
        raise RuntimeError(f"Offline setup needs these pinned packages first: {requirements}")
    command = ["-m", "pip", "install", "--disable-pip-version-check", "--prefer-binary"]
    if all(installed.get(name) for name in stack):
        constraint_path = OUTPUT / "runtime-constraints.txt"
        constraint_path.write_text(
            "\n".join(f"{name}=={installed[name]}" for name in stack) + "\n", encoding="utf-8"
        )
        command += ["--constraint", constraint_path]
    run_python([*command, *requirements])

# From here onward inference never needs to download a model or other assets.
RUN_ENV["HF_HUB_OFFLINE"] = "1"
RUN_ENV["TRANSFORMERS_OFFLINE"] = "1"
probe = run_python(["-c", """
import importlib.metadata, json, torch, torchvision
import solarfilament.inference
names = ('torch','torchvision','numpy','Pillow','scipy','pycocotools','segmentation-models-pytorch')
print(json.dumps({'versions': {n: importlib.metadata.version(n) for n in names},
                  'cuda_available': torch.cuda.is_available(),
                  'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}))
"""], capture=True)
runtime = json.loads(probe)
(OUTPUT / "runtime.json").write_text(json.dumps(runtime, indent=2) + "\n", encoding="utf-8")
print(json.dumps(runtime, indent=2))
if not runtime["cuda_available"]:
    print("CPU selected: full native-resolution inference can be substantially slower.")
freeze = run_python(["-m", "pip", "freeze"], capture=True)
(OUTPUT / "environment-freeze.txt").write_text(freeze, encoding="utf-8")


In [ ]:
# Find images only. Training annotations are neither required nor opened.
configured_data = os.environ.get("MAGFILO_DATA_ROOT")
if configured_data:
    DATA_ROOT = Path(configured_data).expanduser().resolve()
    candidates = [p for p in (DATA_ROOT, DATA_ROOT / "test/test_images", DATA_ROOT / "test_images")
                  if p.is_dir() and any(f.suffix.lower() in {".jpeg", ".jpg", ".png"} for f in p.iterdir() if f.is_file())]
else:
    candidates = list(Path("/kaggle/input").rglob("test_images")) if Path("/kaggle/input").is_dir() else []
candidates = sorted({p.resolve() for p in candidates if p.is_dir()})
if len(candidates) != 1:
    raise RuntimeError(f"Expected one official test-image directory; found {len(candidates)}. Set MAGFILO_DATA_ROOT.")
TEST_IMAGES = candidates[0]
files = sorted(p for p in TEST_IMAGES.iterdir() if p.is_file() and p.suffix.lower() in {".jpeg", ".jpg", ".png"})
expected_ids = [p.stem for p in files]
if len(files) != EXPECTED_IMAGE_COUNT or len(set(expected_ids)) != len(files):
    raise ValueError(f"Expected {EXPECTED_IMAGE_COUNT} distinct test images, found {len(files)}.")
dimension_check = run_python(["-c", """
import json, pathlib, sys
from PIL import Image
root = pathlib.Path(sys.argv[1])
files = sorted(p for p in root.iterdir() if p.is_file() and p.suffix.lower() in {'.jpeg', '.jpg', '.png'})
for path in files:
    with Image.open(path) as image:
        if image.size != (2048, 2048):
            raise ValueError(f'Unexpected test-image dimensions: {path.name} {image.size}')
print(json.dumps({'verified_image_dimensions': len(files)}))
""", TEST_IMAGES], capture=True)
print(dimension_check.strip())
manifest_path = REPO_ROOT / "artifacts/test-images.json"
if manifest_path.is_file():
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    expected_hashes = {item["physical_id"]: item["sha256"] for item in manifest}
    if len(manifest) != EXPECTED_IMAGE_COUNT or set(expected_hashes) != set(expected_ids):
        raise ValueError("Test-image identities differ from the release manifest.")
    for path in files:
        if sha256(path) != expected_hashes[path.stem]:
            raise ValueError(f"Supplied test-image bytes differ from the release manifest: {path.name}")
    print("All test-image SHA-256 hashes match the release manifest.")
else:
    print("Test-image hash manifest is absent; count and output dimensions will still be verified.")
(OUTPUT / "expected-images.json").write_text(json.dumps(expected_ids, indent=2) + "\n", encoding="utf-8")
print(f"Ready to process all {len(files)} official test images in {TEST_IMAGES}.")


In [ ]:
# The final checkpoint contains the frozen inference configuration.
# No training command, threshold tuning, external imagery, or annotation file is used.
run_python(["-m", "solarfilament.cli", "predict",
            "--checkpoint", WEIGHTS, "--image-dir", TEST_IMAGES, "--output-dir", OUTPUT])
run_python(["-m", "solarfilament.cli", "validate",
            "--submission", OUTPUT / "submission.csv", "--image-dir", TEST_IMAGES,
            "--processed", OUTPUT / "processed_images.json"])


In [ ]:
# Compare with the submitted artifact. Numeric drift is never silently accepted.
produced = OUTPUT / "submission.csv"
actual_sha = sha256(produced)
status = {"weight_sha256": provenance["weights_sha256"],
          "expected_submission_sha256": provenance["submission_sha256"],
          "produced_submission_sha256": actual_sha,
          "byte_identical": actual_sha == provenance["submission_sha256"],
          "decoded_mask_equivalent": False}
reference = REPO_ROOT / "submissions/final.csv"
if status["byte_identical"]:
    status["decoded_mask_equivalent"] = True
elif reference.is_file():
    if sha256(reference) != provenance["submission_sha256"]:
        raise ValueError("Reference CSV checksum differs from the submitted artifact in provenance.")
    # Compare each image's multiset of decoded instance masks, ignoring CSV order
    # and arbitrary instance suffixes. Preserve instance boundaries, not just union.
    comparison_code = """
import csv, hashlib, json, sys
from collections import defaultdict
from solarfilament.rle import decode_rle
def signatures(path):
    result = defaultdict(list)
    with open(path, newline='', encoding='utf-8') as handle:
        for row in csv.DictReader(handle):
            stem = row['filament_id'].split('_', 1)[0]
            mask = decode_rle(row['segmentation_rle'], (2048, 2048))
            result[stem].append(hashlib.sha256(mask.tobytes(order='C')).hexdigest())
    return {stem: sorted(values) for stem, values in result.items()}
print(json.dumps({'equivalent': signatures(sys.argv[1]) == signatures(sys.argv[2])}))
"""
    comparison = json.loads(run_python(["-c", comparison_code, produced, reference], capture=True))
    status["decoded_mask_equivalent"] = comparison["equivalent"]
else:
    status["reason"] = "Byte hash differs and submissions/final.csv is unavailable for decoded-mask comparison."
(OUTPUT / "reproduction.json").write_text(json.dumps(status, indent=2) + "\n", encoding="utf-8")
print(json.dumps(status, indent=2))
if not status["decoded_mask_equivalent"]:
    raise RuntimeError("Exact reproduction is not established. Inspect reproduction.json and runtime.json; do not claim equivalence.")
print(f"Reproduction verified. Predictions: {produced}")


The saved `inference.json`, `runtime.json`, environment freeze and `reproduction.json` document the actual run. Byte equality is preferred; the fallback comparison requires exact equality of each decoded instance mask and does not relax pixel tolerances. If GPU/library differences alter mask pixels, execution reports the discrepancy and requires investigation. Kaggle acceptance, final selection, public repository access and organizer-form receipt must be checked separately.
